# ⬜ Capa Silver: Limpieza, Calidad de Datos y Tipado Estricto
Este notebook consume las tablas crudas de la capa Bronze, extrae los valores embebidos en XML/JSON, estandariza los esquemas a un formato común y elimina duplicados para garantizar la idempotencia del pipeline.

In [8]:
import xml.etree.ElementTree as ET
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime, timedelta

def transformar_xml_entsoe(raw_table_name, pais_nombre):
    raw_df = spark.read.table(raw_table_name).collect()
    xml_content = raw_df[0]["raw_xml"]
    root = ET.fromstring(xml_content)
    
    ns = {'ns': 'urn:entsoe.eu:wgedi:components'}
    registros = []
    
    for ts in root.findall('.//ns:TimeSeries', ns):
        start_str = ts.find('.//ns:start', ns).text
        start_dt = datetime.strptime(start_str, "%Y-%m-%dT%H:%MZ") # Detección de huso UTC nativo
        
        for point in ts.findall('.//ns:Point', ns):
            pos = int(point.find('ns:position', ns).text)
            precio = float(point.find('ns:price.amount', ns).text)
            # PT15M asumido fijo para Day Ahead en ENTSO-E
            timestamp_registro = start_dt + timedelta(minutes=(pos - 1) * 15)
            registros.append((timestamp_registro.strftime("%Y-%m-%d %H:%M:%S"), precio, pais_nombre, "ENTSO-E"))
            
    schema = StructType([
        StructField("timestamp_utc", StringType(), True),
        StructField("precio_mwh", DoubleType(), True),
        StructField("pais", StringType(), True),
        StructField("fuente", StringType(), True)
    ])
    return spark.createDataFrame(registros, schema)

StatementMeta(, 76491fc3-df18-478f-95e8-907b55cb181e, 10, Finished, Available, Finished, False)

In [9]:
print("🧹 Transformando y normalizando estructuras de datos...")

# Mercados ENTSO-E (España y Rumanía)
df_es_silver = transformar_xml_entsoe("bronze_raw_espana", "España")
df_ro_silver = transformar_xml_entsoe("bronze_raw_rumania", "Rumania")

# Alemania — explode del array de arrays [timestamp_ms, precio]
df_de_raw = spark.read.table("bronze_raw_alemania")

df_de_silver = df_de_raw.select(
    F.explode(F.col("series")).alias("datos")
).select(
    # SMARD devuelve timestamps Unix en milisegundos
    F.to_timestamp(F.col("datos")[0] / 1000).alias("timestamp_utc"),
    F.col("datos")[1].cast("double").alias("precio_mwh"),
    F.lit("Alemania").alias("pais"),
    F.lit("SMARD").alias("fuente")
)

StatementMeta(, 76491fc3-df18-478f-95e8-907b55cb181e, 11, Finished, Available, Finished, False)

🧹 Transformando y normalizando estructuras de datos...


In [10]:
def guardar_tabla_silver(df, nombre_tabla):
    df_limpio = df.select(
        F.col("timestamp_utc").cast("string").alias("timestamp_utc"),
        F.col("precio_mwh").cast("double").alias("precio_mwh"),
        F.col("pais").cast("string").alias("pais"),
        F.col("fuente").cast("string").alias("fuente")
    ).dropDuplicates(["timestamp_utc", "pais"])
    
    # overwriteSchema evita conflictos de metadatos entre ejecuciones
    df_limpio.write.format("delta") \
             .mode("overwrite") \
             .option("overwriteSchema", "true") \
             .saveAsTable(nombre_tabla)

print("✔️ Función de guardado resiliente configurada.")

StatementMeta(, 76491fc3-df18-478f-95e8-907b55cb181e, 12, Finished, Available, Finished, False)

✔️ Función de guardado resiliente configurada.
